<a href="https://colab.research.google.com/github/Chosencodes/Medical-Imaging-Projects/blob/main/Cardiac_Detection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Preprocessing**

In [ ]:
!pip install pydicom

In [ ]:
from pathlib import Path
import pydicom
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import torch
import albumentations as A
from albumentations.pytorch import ToTensorV2

In [ ]:
!mkdir -p ~/.kaggle
!cp "kaggle (1).json" ~/.kaggle/kaggle.json
!chmod 600 ~/.kaggle/kaggle.json

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!kaggle competitions download -c rsna-pneumonia-detection-challenge

In [ ]:
!unzip -q rsna-pneumonia-detection-challenge.zip -d /content/rsna

In [ ]:
labels = pd.read_csv("/content/drive/MyDrive/05-Detection/rsna_heart_detection.csv")

In [ ]:
labels.head(3)

In [ ]:
ROOT_PATH = Path("/content/rsna/stage_2_train_images/")
SAVE_PATH = Path("Processed-Heart-Detection")


In [ ]:
import os

all_files = set(os.listdir("/content/rsna/stage_2_train_images/"))

labels = labels[labels["name"].apply(lambda x: x + ".dcm" in all_files)].reset_index(drop=True)

In [ ]:
fig,axis = plt.subplots(2,2,figsize=(10,10))
c = 0

for i in range(2):
  for j in range(2):
    data = labels.iloc[c]
    patient_id = data["name"]
    dcm_path =ROOT_PATH/str(patient_id)
    dcm_path = dcm_path.with_suffix('.dcm')
    dcm = pydicom.dcmread(dcm_path).pixel_array

    dcm_array = cv2.resize(dcm,(224,224))

    x = data["x0"]
    y = data["y0"]
    w = data["w"]
    h = data["h"]

    axis[i][j].imshow(dcm_array,cmap="bone")
    rect = patches.Rectangle((x,y),w,h,linewidth=1,edgecolor="r",facecolor="none")
    axis[i][j].add_patch(rect)
    c+=1



In [ ]:
sums = 0
sums_squared = 0
train_ids = []
val_ids = []

for c, patient_id in enumerate(list(labels.name)):
  dcm_path = ROOT_PATH/str(patient_id)
  dcm_path = dcm_path.with_suffix('.dcm')
  dcm = pydicom.dcmread(dcm_path).pixel_array

  dcm_array = (cv2.resize(dcm,(224,224)) / 255).astype(np.float16)

  train_or_val = "train" if c < 400 else "val"

  if train_or_val == "train":
    train_ids.append(patient_id)
  else:
    val_ids.append(patient_id)

  current_save_path = SAVE_PATH/train_or_val/patient_id
  current_save_path.mkdir(parents=True, exist_ok=True)
  np.save(current_save_path/patient_id,dcm_array)

  normalizer = 224*224
  if train_or_val == "train":
    sums += np.sum(dcm_array) / normalizer
    sums_squared += (dcm_array ** 2).sum() / normalizer

In [ ]:
np.save("Processed-Heart-Detection/train_subjects_det",train_ids)
np.save("Processed-Heart-Detection/val_subjects_det",val_ids)

In [ ]:
mean = sums / len(train_ids)
std = np.sqrt((sums_squared / len(train_ids)) - mean ** 2)

In [ ]:
mean,std

# **DATASET**